<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.3-admin-observability/practice/GCP_Capstone_12.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 12.3 — Admin Dashboard & Observability

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup — install, authenticate, initialise clients

Run this first. It installs the SDKs the lab uses, authenticates via Application Default Credentials (Colab), sets the project/region/bucket env vars every later cell reads, and initialises the unified Gemini client used in Exercise 4.

**Replace `documind-ai-YOUR-ID` with your own project id.** Infra cells (Terraform, `gcloud`, `bq`) are shown exactly as they run in production — they need a real project and IAM to complete; on a fresh Colab they write the config files and you apply them against your project.

In [ ]:
%%bash
pip install -q \
  google-genai \
  google-cloud-dlp \
  google-cloud-firestore \
  google-cloud-storage \
  google-cloud-bigquery \
  pandas plotly streamlit
echo 'deps installed'

In [ ]:
# One-time setup — auth, project, region, clients
import os

PROJECT_ID    = "documind-ai-YOUR-ID"   # <-- replace with your project id
COURSE_REGION = "us-central1"           # course examples
INDIA_REGION  = "asia-south1"           # India production (DLP + BQ dataset live here)

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["AUDIT_BUCKET"]         = f"{PROJECT_ID}-audit"

# Colab auth -> Application Default Credentials (never API keys)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab ADC")
except ImportError:
    print("Not on Colab — assuming ADC is already configured (gcloud auth application-default login)")

# Unified Gemini SDK on Vertex — used in Exercise 4 to prove redacted prompts still answer
from google import genai
from google.genai import types
client = genai.Client(enterprise=True, project=PROJECT_ID, location="global")  # Gemini 3.x generation is served from the global endpoint
print("genai client ready:", PROJECT_ID, "(generation endpoint: global)")

## Exercise 1: BigQuery dataset + Log Sink

**Difficulty:** Easy

Terraform a `documind_observability` dataset in asia-south1 with 90-day table expiration. Create a Log Sink from Cloud Logging filtering `jsonPayload.event="query"` into BigQuery with partitioned tables + a unique writer identity.

**Steps**
1. Write a `google_bigquery_dataset` in `var.india_region` with `default_table_expiration_ms = 7776000000` (90 days).
2. Write a `google_logging_project_sink` filtered to the `documind-api` Cloud Run service and `jsonPayload.event = "query"`, destined for the dataset, with `unique_writer_identity = true` and `use_partitioned_tables = true`.
3. Grant the sink's writer identity `roles/bigquery.dataEditor` on the dataset.
4. Apply, then verify with `bq ls` and a test `/v1/query`.

**Expected behaviour:** `bq ls` shows the dataset. Send a test `/v1/query`; within 30s a row appears in the `run_googleapis_com_stdout` partition.

In [ ]:
# Cell 1 (lesson): sink.tf — structured API logs -> BigQuery, partitioned, 90-day retention
SINK_TF = '''
resource "google_bigquery_dataset" "observability" {
  dataset_id    = "documind_observability"
  location      = var.india_region
  description   = "API structured logs, DLP findings, tenant rollups"
  default_table_expiration_ms = 7776000000   # 90 days for raw logs
  delete_contents_on_destroy  = false
}

resource "google_logging_project_sink" "api_to_bq" {
  name        = "documind-api-to-bq"
  destination = "bigquery.googleapis.com/projects/${var.project_id}/datasets/${google_bigquery_dataset.observability.dataset_id}"
  filter      = <<EOT
    resource.type = "cloud_run_revision"
    resource.labels.service_name = "documind-api"
    jsonPayload.event = "query"
  EOT
  unique_writer_identity = true
  bigquery_options { use_partitioned_tables = true }
}

# Sink identity needs BQ data editor
resource "google_bigquery_dataset_iam_member" "sink_writer" {
  dataset_id = google_bigquery_dataset.observability.dataset_id
  role       = "roles/bigquery.dataEditor"
  member     = google_logging_project_sink.api_to_bq.writer_identity
}
'''
with open('sink.tf', 'w') as f: f.write(SINK_TF)
print('sink.tf written')
print()
print('Partitioned by day on timestamp. One row per /v1/query call.')
print('Columns auto-derived from jsonPayload: tenant, user, latency_ms, tokens_in/out, confidence, answerable')

In [ ]:
%%bash
# Apply the sink, then confirm the dataset + a fresh partition exist
terraform init -input=false
terraform apply -auto-approve \
  -var project_id="$GOOGLE_CLOUD_PROJECT" \
  -var india_region=asia-south1

bq ls --project_id="$GOOGLE_CLOUD_PROJECT" documind_observability

# After sending a test /v1/query, a row lands within ~30s:
bq query --use_legacy_sql=false --project_id="$GOOGLE_CLOUD_PROJECT" \
  "SELECT COUNT(*) AS rows_last_hour
   FROM documind_observability.run_googleapis_com_stdout
   WHERE timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)"

## Exercise 2: `tenant_daily` materialised view

**Difficulty:** Easy

`CREATE MATERIALIZED VIEW` with day + tenant grouping, counts, token sums, and `APPROX_QUANTILES` at offsets 50/95/99. `PARTITION BY day`, `CLUSTER BY tenant`. Measure query cost before vs after.

**Steps**
1. Aggregate `run_googleapis_com_stdout` grouped by `DATE(timestamp, "Asia/Kolkata")` and `jsonPayload.tenant`.
2. Emit `queries`, `unanswerable`, `tokens_in`, `tokens_out`, and p50/p95/p99 latency via `APPROX_QUANTILES(..., 100)[OFFSET(n)]`.
3. `PARTITION BY day`, `CLUSTER BY tenant`.
4. Dry-run a dashboard query against the raw table vs the MV and compare bytes scanned.

**Expected behaviour:** Raw scan ~500MB; the materialised-view query scans ~10MB. Dashboard load drops from 2s to 200ms.

In [ ]:
# Cell 2 (lesson): tenant_daily.sql — daily rollup materialised view
DAILY_ROLLUP_SQL = '''
CREATE MATERIALIZED VIEW IF NOT EXISTS `documind_observability.tenant_daily`
PARTITION BY day
CLUSTER BY tenant
AS
SELECT
  DATE(timestamp, "Asia/Kolkata") AS day,
  jsonPayload.tenant AS tenant,
  COUNT(*) AS queries,
  COUNTIF(jsonPayload.answerable = false) AS unanswerable,
  SUM(CAST(jsonPayload.tokens_in  AS INT64)) AS tokens_in,
  SUM(CAST(jsonPayload.tokens_out AS INT64)) AS tokens_out,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(50)] AS p50_ms,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(95)] AS p95_ms,
  APPROX_QUANTILES(CAST(jsonPayload.latency_ms AS INT64), 100)[OFFSET(99)] AS p99_ms
FROM `documind_observability.run_googleapis_com_stdout`
WHERE jsonPayload.event = "query"
GROUP BY day, tenant;
'''
with open('tenant_daily.sql', 'w') as f: f.write(DAILY_ROLLUP_SQL)
print('tenant_daily.sql written')
print()
print('Materialised view refreshes automatically. Query cost for dashboard ~10MB scanned.')

In [ ]:
%%bash
# Create the MV, then compare bytes scanned raw vs materialised (--dry_run reports the estimate)
bq query --use_legacy_sql=false --project_id="$GOOGLE_CLOUD_PROJECT" < tenant_daily.sql

echo '--- raw table (expensive) ---'
bq query --use_legacy_sql=false --dry_run --project_id="$GOOGLE_CLOUD_PROJECT" \
  "SELECT jsonPayload.tenant, COUNT(*)
   FROM documind_observability.run_googleapis_com_stdout
   WHERE jsonPayload.event = 'query' GROUP BY 1"

echo '--- materialised view (cheap) ---'
bq query --use_legacy_sql=false --dry_run --project_id="$GOOGLE_CLOUD_PROJECT" \
  "SELECT tenant, SUM(queries) FROM documind_observability.tenant_daily GROUP BY 1"

## Exercise 3: DLP inspect with India info types

**Difficulty:** Medium

Write `inspect_and_log(chunk_id, tenant_id, text)` using `INDIA_AADHAAR_INDIVIDUAL`, `INDIA_PAN_INDIVIDUAL`, `EMAIL_ADDRESS`, `PHONE_NUMBER` with `min_likelihood=LIKELY` and `include_quote=False`. Store findings in Firestore `dlp_findings`. Test on `"Raj Kumar, raj@ex.in, PAN ABCDE1234F, Aadhaar 1234 5678 9012"`.

**Steps**
1. Build the info-type list (add PERSON_NAME so the name is caught too).
2. Call `inspect_content` with `include_quote=False` so raw PII never lands in the log.
3. Persist finding metadata (info_type, likelihood, offset) to Firestore `dlp_findings` — never the text.
4. Return `{"has_pii": ..., "types": [...]}` and test on the sample string.

**Expected behaviour:** Returns `{"has_pii": true, "types": ["EMAIL_ADDRESS","INDIA_AADHAAR_INDIVIDUAL","INDIA_PAN_INDIVIDUAL","PERSON_NAME"]}`. The Firestore doc holds no raw text, only finding metadata.

In [ ]:
# Cell 3 (lesson): dlp.py — inspect_and_log() + redact() with India info types
DLP_PY = '''
import os
from google.cloud import dlp_v2
from google.cloud import firestore

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
INDIA_INFO_TYPES = [
    {"name": "INDIA_AADHAAR_INDIVIDUAL"},
    {"name": "INDIA_PAN_INDIVIDUAL"},
    {"name": "INDIA_GST_INDIVIDUAL"},
    {"name": "EMAIL_ADDRESS"},
    {"name": "PHONE_NUMBER"},
    {"name": "PERSON_NAME"},
    {"name": "DATE_OF_BIRTH"},
]
MIN_LIKELIHOOD = "LIKELY"

_dlp = dlp_v2.DlpServiceClient()
_fs  = firestore.Client(project=PROJECT)

def inspect_and_log(chunk_id: str, tenant_id: str, text: str) -> dict:
    parent = f"projects/{PROJECT}/locations/asia-south1"
    resp = _dlp.inspect_content(
        request={
            "parent": parent,
            "inspect_config": {
                "info_types": INDIA_INFO_TYPES,
                "min_likelihood": MIN_LIKELIHOOD,
                "include_quote": False,  # DO NOT store PII in audit logs
                "limits": {"max_findings_per_request": 50},
            },
            "item": {"value": text},
        })
    findings = [{"info_type": f.info_type.name, "likelihood": f.likelihood.name,
                 "offset": f.location.byte_range.start if f.location.byte_range else None}
                for f in resp.result.findings]
    if findings:
        _fs.collection("dlp_findings").add({
            "chunk_id": chunk_id, "tenant_id": tenant_id,
            "findings": findings, "count": len(findings),
            "scanned_at": firestore.SERVER_TIMESTAMP,
        })
    return {"has_pii": bool(findings), "types": sorted({f["info_type"] for f in findings})}

def redact(text: str) -> str:
    """Use BEFORE sending to external providers (OpenAI fallback)."""
    parent = f"projects/{PROJECT}/locations/asia-south1"
    resp = _dlp.deidentify_content(request={
        "parent": parent,
        "inspect_config": {"info_types": INDIA_INFO_TYPES, "min_likelihood": "POSSIBLE"},
        "deidentify_config": {"info_type_transformations": {
            "transformations": [{
                "primitive_transformation": {
                    "replace_with_info_type_config": {}
                }}]}},
        "item": {"value": text},
    })
    return resp.item.value  # "Send to [EMAIL_ADDRESS] by [DATE_OF_BIRTH]"
'''
with open('dlp.py', 'w') as f: f.write(DLP_PY)
print('dlp.py written')
print()
print('inspect_and_log() used on every uploaded chunk during ingestion (Module 11 pipeline hook)')
print('redact() used on every prompt routed to non-India providers (LiteLLM pre-request hook)')

In [ ]:
# Test inspect_and_log on the lab's sample string
from dlp import inspect_and_log

result = inspect_and_log(
    chunk_id="test_chunk_001",
    tenant_id="tenant-acme",
    text="Raj Kumar, raj@ex.in, PAN ABCDE1234F, Aadhaar 1234 5678 9012",
)
print(result)
# -> {'has_pii': True, 'types': ['EMAIL_ADDRESS','INDIA_AADHAAR_INDIVIDUAL','INDIA_PAN_INDIVIDUAL','PERSON_NAME']}
# The Firestore dlp_findings doc holds only info_type/likelihood/offset — never the raw text.

## Exercise 4: DLP redact for cross-border routing

**Difficulty:** Medium

Implement `redact(text)` using `deidentify_content` with `replace_with_info_type_config`. Hook it into Module 11's LiteLLM request middleware so CONFIDENTIAL-tier queries are redacted before non-India provider calls. Confirm the response still makes sense with placeholders.

**Steps**
1. Reuse `redact()` from `dlp.py` (Exercise 3) — it swaps each finding for `[INFO_TYPE]`.
2. Redact the prompt before it leaves India, then send the placeholdered prompt to the model.
3. Confirm the answer is still coherent using only the placeholders.

**Expected behaviour:** `"Contact raj@ex.in"` → `"Contact [EMAIL_ADDRESS]"`. The LLM still answers coherently using the placeholder.

In [ ]:
# redact() lives in dlp.py (written in Exercise 3). Prove a redacted prompt still answers.
from dlp import redact

sample = "Summarise this support ticket: contact raj@ex.in about a refund; his PAN is ABCDE1234F."
clean = redact(sample)
print("Redacted prompt sent cross-border:\n ", clean)

# This is the LiteLLM non-India hop — the model sees placeholders, not PII
resp = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=f"You are a support assistant.\n{clean}\nWhat action is the customer requesting?",
)
print("\nModel answer (coherent despite redaction):\n ", resp.text)

## Exercise 5: Dual-write audit log (GCS + Firestore)

**Difficulty:** Medium

Write `emit(action, actor, target, meta)`. Reject unregistered actions (an `AUDIT_ACTIONS` whitelist). Write JSON to `$PROJECT-audit/YYYY/MM/DD/tenant/action-uuid.json`. Dual-write to Firestore `audit_index`. Add a Firestore TTL policy of 14 days.

**Steps**
1. Whitelist the allowed action strings; `assert` any unknown action.
2. Build the event (`id`, `ts`, `action`, `actor`, `target`, `meta`) and write it to the date-partitioned GCS path — GCS is the 5-year locked source of truth.
3. Dual-write to Firestore `audit_index` with an `expire_at` Timestamp so a 14-day TTL policy can auto-purge the hot index.
4. Enable the TTL policy on the `expire_at` field.

**Expected behaviour:** `gsutil ls gs://$PROJECT-audit/$(date +%Y)/` shows today's events. The `audit_index` collection has a TTL field; 15-day-old docs auto-delete.

In [ ]:
# Cell 4 (lesson): audit.py — append-only audit emitter, GCS source-of-truth + Firestore hot index
# Light adaptation vs the lesson: the Firestore doc carries an `expire_at` Timestamp so the
# 14-day TTL policy in the bash cell below has a real field to act on.
AUDIT_PY = '''
import os, uuid, json
from datetime import datetime, timezone, timedelta
from google.cloud import storage, firestore

AUDIT_BUCKET = storage.Client().bucket(os.environ["AUDIT_BUCKET"])   # 5y LOCKED retention (12.1)
_fs = firestore.Client(project=os.environ["GOOGLE_CLOUD_PROJECT"])

AUDIT_ACTIONS = {
    "user.login", "user.logout",
    "doc.upload", "doc.delete", "doc.download",
    "query.submit", "query.export",
    "admin.view_tenant", "admin.rotate_key",
    "tenant.create", "tenant.suspend",
    "dlp.finding", "consent.grant", "consent.revoke",
}

def emit(action: str, actor: dict, target: dict, meta: dict | None = None):
    assert action in AUDIT_ACTIONS, f"unregistered audit action: {action}"
    now = datetime.now(timezone.utc)
    event = {
        "id": str(uuid.uuid4()),
        "ts": now.isoformat(),
        "action": action,
        "actor": actor,     # {email, sub, tenant_id, ip}
        "target": target,   # {type, id, tenant_id}
        "meta": meta or {},
    }
    # Path: year/month/day/tenant/action-uuid.json
    blob = AUDIT_BUCKET.blob(
        f"{now.year}/{now.month:02d}/{now.day:02d}/{actor.get('tenant_id','_')}/"
        f"{action}-{event['id']}.json")
    blob.upload_from_string(json.dumps(event, separators=(',',':')),
                            content_type="application/json")
    # Also index in Firestore for fast dashboard filter (14-day TTL on expire_at)
    _fs.collection("audit_index").document(event["id"]).set({
        **event,
        "expire_at": now + timedelta(days=14),   # Firestore TTL field (Timestamp)
    })
    return event["id"]
'''
with open('audit.py', 'w') as f: f.write(AUDIT_PY)
print('audit.py written')
print()
print('Dual-write pattern: GCS is source of truth (5y retention, locked).')
print('Firestore audit_index is a 14-day hot index for the dashboard.')

In [ ]:
# Emit a test event -> writes to GCS + Firestore, returns the event id
from audit import emit

eid = emit(
    "doc.upload",
    {"email": "alice@acme.in", "sub": "u_1", "tenant_id": "tenant-acme", "ip": "1.2.3.4"},
    {"type": "doc", "id": "doc_9f23", "tenant_id": "tenant-acme"},
    {"filename": "invoice.pdf"},
)
print("audit event id:", eid)

# Rejects anything not on the whitelist:
try:
    emit("doc.shred", {"tenant_id": "tenant-acme"}, {"type": "doc", "id": "x"})
except AssertionError as e:
    print("rejected:", e)

In [ ]:
%%bash
# Confirm today's events landed in GCS, then enable the 14-day Firestore TTL on expire_at
gsutil ls "gs://$GOOGLE_CLOUD_PROJECT-audit/$(date +%Y)/$(date +%m)/$(date +%d)/tenant-acme/"

gcloud firestore fields ttls update expire_at \
  --collection-group=audit_index \
  --enable-ttl \
  --project="$GOOGLE_CLOUD_PROJECT"
# Docs whose expire_at is >14 days old are auto-deleted by Firestore's TTL background job.

## Exercise 6: Streamlit admin tabs

**Difficulty:** Medium

Build `admin_dashboard.py` with four tabs: Usage (`tenant_daily` chart), Tenants (create-tenant form + list), Audit Log (filter by action), DLP (findings histogram). Every action that mutates state calls `audit.emit()`.

**Steps**
1. Cache the `tenant_daily` query (`@st.cache_data(ttl=300)`) so the dashboard reads the MV, not the raw table.
2. Usage tab: KPI metrics + per-day queries bar + latency-percentile lines.
3. Tenants tab: list from Firestore + a create form that calls `emit("tenant.create", ...)`.
4. Audit tab: filter the `audit_index` by action. DLP tab: histogram of findings by type.

**Expected behaviour:** All four tabs render. Creating a tenant emits a `tenant.create` audit event. The dashboard loads in < 500ms thanks to the MV cache.

In [ ]:
# Cell 5 (lesson): admin_dashboard.py — Streamlit admin tabs (Usage / Tenants / Audit / DLP)
ADMIN_PY = '''
import os, pandas as pd, plotly.express as px, streamlit as st
from google.cloud import bigquery, firestore

_bq = bigquery.Client()
_fs = firestore.Client(project=os.environ["GOOGLE_CLOUD_PROJECT"])
DATASET = "documind_observability"

@st.cache_data(ttl=300)
def tenant_daily(tenant: str | None, days: int = 30) -> pd.DataFrame:
    where = "day >= DATE_SUB(CURRENT_DATE('Asia/Kolkata'), INTERVAL @days DAY)"
    params = [bigquery.ScalarQueryParameter("days", "INT64", days)]
    if tenant:
        where += " AND tenant = @t"
        params.append(bigquery.ScalarQueryParameter("t", "STRING", tenant))
    sql = f"SELECT * FROM `{DATASET}.tenant_daily` WHERE {where} ORDER BY day"
    return _bq.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

def usage_tab():
    st.subheader("Usage (last 30 days)")
    tenant = st.selectbox("Tenant filter", ["All"] + list_tenants()) or "All"
    df = tenant_daily(None if tenant == "All" else tenant)
    if df.empty: st.info("No queries in window."); return
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total queries", f"{df['queries'].sum():,}")
    col2.metric("Tokens (M)", f"{(df['tokens_in'].sum()+df['tokens_out'].sum())/1e6:.2f}")
    col3.metric("p95 latency", f"{df['p95_ms'].median():.0f} ms")
    unans_pct = 100*df["unanswerable"].sum()/max(1, df["queries"].sum())
    col4.metric("Unanswerable", f"{unans_pct:.1f} %")
    st.plotly_chart(px.bar(df, x="day", y="queries", color="tenant",
                           title="Queries per day"))
    st.plotly_chart(px.line(df, x="day", y=["p50_ms","p95_ms","p99_ms"],
                            title="Latency percentiles (ms)"))

def tenants_tab():
    st.subheader("Tenants")
    rows = []
    for t in _fs.collection("tenants").stream():
        d = t.to_dict(); d["id"] = t.id; rows.append(d)
    df = pd.DataFrame(rows)
    st.dataframe(df, use_container_width=True)
    with st.expander("Create tenant"):
        tid = st.text_input("Tenant ID")
        tier = st.selectbox("Tier", ["free","pro","enterprise"])
        quota = st.number_input("Monthly budget (USD)", 10, 10000, 50)
        if st.button("Create") and tid:
            _fs.collection("tenants").document(tid).set({
                "tier": tier, "max_budget_usd": quota,
                "created_at": firestore.SERVER_TIMESTAMP,
            })
            from audit import emit
            emit("tenant.create", st.session_state.user,
                 {"type":"tenant","id":tid}, {"tier":tier,"budget":quota})
            st.success(f"Created {tid}"); st.rerun()

def audit_tab():
    st.subheader("Audit log (last 14 days)")
    action = st.selectbox("Action", ["all","user.login","doc.upload","doc.delete",
                                     "admin.rotate_key","tenant.suspend"])
    q = _fs.collection("audit_index").order_by("ts", direction="DESCENDING").limit(500)
    if action != "all": q = q.where("action", "==", action)
    rows = [d.to_dict() for d in q.stream()]
    st.dataframe(pd.DataFrame(rows), use_container_width=True)
    st.caption("Full 5-year audit is in GCS. Firestore index is hot 14 days only.")

def dlp_tab():
    st.subheader("DLP findings")
    q = _fs.collection("dlp_findings").order_by("scanned_at", direction="DESCENDING").limit(200)
    rows = [d.to_dict() for d in q.stream()]
    df = pd.DataFrame(rows)
    if df.empty: st.info("No findings."); return
    st.metric("Chunks with findings", len(df))
    flat = []
    for _, r in df.iterrows():
        for f in r["findings"]:
            flat.append({"tenant": r["tenant_id"], "type": f["info_type"],
                         "likelihood": f["likelihood"]})
    st.plotly_chart(px.histogram(pd.DataFrame(flat), x="type", color="likelihood",
                                 title="PII findings by type"))

def list_tenants() -> list[str]:
    return sorted(t.id for t in _fs.collection("tenants").stream())

def admin_page(user):
    st.title("\U0001F6E0 DocuMind Admin")
    tab1, tab2, tab3, tab4 = st.tabs(["Usage","Tenants","Audit Log","DLP"])
    with tab1: usage_tab()
    with tab2: tenants_tab()
    with tab3: audit_tab()
    with tab4: dlp_tab()
'''
with open('admin_dashboard.py', 'w') as f: f.write(ADMIN_PY)
print('admin_dashboard.py written')

## Exercise 7: Monitoring alerts (SLO + product signal)

**Difficulty:** Challenge

Terraform two alert policies: (A) Cloud Run p95 latency > 3s for 5 min, (B) `unanswerable_rate` > 0.20 for 30 min. Wire to a PagerDuty notification channel. Synthesise load with `hey -n 200 -c 20`; confirm policy A fires when you artificially add `time.sleep(5)` to the handler.

**Steps**
1. Define a PagerDuty `google_monitoring_notification_channel`.
2. Policy A: `ALIGN_PERCENTILE_95` on `run.googleapis.com/request_latencies` > 3000ms for 300s.
3. Policy B: `unanswerable_rate` log-based metric > 0.20 for 1800s (a product signal infra checks miss).
4. Apply, generate load with `hey`, and watch for the page.

**Expected behaviour:** `gcloud alpha monitoring policies list` shows 2 policies ENABLED. The synthetic slow path triggers a PagerDuty page within 6-7 min.

In [ ]:
# Cell 6 (lesson): alerts.tf — two Cloud Monitoring alert policies (infra SLO + product signal)
ALERTS_TF = '''
resource "google_monitoring_notification_channel" "oncall" {
  display_name = "DocuMind on-call"
  type         = "pagerduty"
  labels       = { service_key = var.pagerduty_key }
  sensitive_labels { auth_token { value = var.pagerduty_key } }
}

# SLO: p95 /v1/query < 3s over 30-min rolling window
resource "google_monitoring_alert_policy" "api_latency" {
  display_name = "API p95 latency > 3s"
  combiner     = "OR"
  conditions {
    display_name = "p95 > 3s for 5 minutes"
    condition_threshold {
      filter     = "resource.type=\\"cloud_run_revision\\" AND resource.labels.service_name=\\"documind-api\\" AND metric.type=\\"run.googleapis.com/request_latencies\\""
      comparison = "COMPARISON_GT"
      threshold_value = 3000
      duration   = "300s"
      aggregations {
        alignment_period     = "60s"
        per_series_aligner   = "ALIGN_PERCENTILE_95"
        cross_series_reducer = "REDUCE_MEAN"
      }
    }
  }
  notification_channels = [google_monitoring_notification_channel.oncall.id]
  alert_strategy { auto_close = "1800s" }
}

# Unanswerable rate spike (product signal, not infra)
resource "google_monitoring_alert_policy" "unanswerable_rate" {
  display_name = "Unanswerable rate > 20% for a tenant"
  combiner     = "OR"
  conditions {
    display_name = "unanswerable_rate > 0.20"
    condition_threshold {
      filter     = "metric.type=\\"logging.googleapis.com/user/documind/unanswerable_rate\\""
      comparison = "COMPARISON_GT"
      threshold_value = 0.20
      duration   = "1800s"
      aggregations {
        alignment_period     = "300s"
        per_series_aligner   = "ALIGN_MEAN"
      }
    }
  }
  notification_channels = [google_monitoring_notification_channel.oncall.id]
}
'''
with open('alerts.tf', 'w') as f: f.write(ALERTS_TF)
print('alerts.tf written')
print()
print('Two policies: infra SLO (p95 latency) + product signal (unanswerable rate).')
print('Product signal catches retrieval regressions that would pass every infra check.')

In [ ]:
%%bash
# Apply the policies, list them, then synthesise load to trip policy A (needs a sleep(5) in the handler)
terraform apply -auto-approve -var pagerduty_key="$PAGERDUTY_KEY"

gcloud alpha monitoring policies list \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --format='table(displayName, enabled)'

# Drive traffic; with time.sleep(5) added to /v1/query, p95 crosses 3s and PagerDuty pages in ~6 min
hey -n 200 -c 20 -m POST -H "Authorization: Bearer $TOK" \
  https://documind-api-xxx.run.app/v1/query

## Exercise 8: Deploy admin UI with IAP group gating

**Difficulty:** Challenge

Deploy `documind-admin` on Cloud Run with `documind-admin-sa` + IAP. Restrict access to `group:admin-group@documind.ai`. Verify a non-admin user gets 403 at IAP. Verify an admin user sees all 4 tabs. Grant admin-sa `bigquery.dataViewer` + `bigquery.jobUser`.

**Steps**
1. Deploy a separate `documind-admin` Cloud Run service (same image as 12.4) with `--no-allow-unauthenticated` and the admin SA.
2. Enable IAP and bind `group:admin-group@documind.ai` to `roles/iap.httpsResourceAccessor`.
3. Grant the admin SA `bigquery.dataViewer` + `bigquery.jobUser` so the dashboard can read the MV.
4. Test: non-admin → IAP 403; admin → all four tabs live.

**Expected behaviour:** The IAP policy shows the `group:admin-group` principal. Non-admin: IAP 403 page. Admin: dashboard with usage/tenants/audit/dlp tabs live.

In [ ]:
%%bash
# Cell 7 (lesson): deploy documind-admin as a separate IAP-gated Cloud Run service + scoped IAM
# Same Streamlit image as 12.4; the Admin tabs are gated by IAP group membership, not app code.

gcloud run deploy documind-admin \
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/ui:$GIT_SHA \
  --region=us-central1 --no-allow-unauthenticated \
  --ingress=internal-and-cloud-load-balancing \
  --service-account=documind-admin-sa@$PROJECT.iam.gserviceaccount.com \
  --set-env-vars="ADMIN_EMAILS=alice@documind.ai,bob@documind.ai,AUDIT_BUCKET=$PROJECT-audit" \
  --set-secrets="COOKIE_SECRET=cookie-secret:latest" \
  --session-affinity --cpu-boost

# IAP group restricted to admin-group@documind.ai
gcloud beta run services update documind-admin --region=us-central1 --iap
gcloud beta iap web add-iam-policy-binding \
  --resource-type=cloud-run --service=documind-admin --region=us-central1 \
  --member="group:admin-group@documind.ai" \
  --role="roles/iap.httpsResourceAccessor"

# Admin SA needs BQ data viewer + job user to read the tenant_daily MV
gcloud projects add-iam-policy-binding $PROJECT \
  --member="serviceAccount:documind-admin-sa@$PROJECT.iam.gserviceaccount.com" \
  --role="roles/bigquery.dataViewer"
gcloud projects add-iam-policy-binding $PROJECT \
  --member="serviceAccount:documind-admin-sa@$PROJECT.iam.gserviceaccount.com" \
  --role="roles/bigquery.jobUser"

# Verify: a non-admin hitting the URL gets an IAP 403; an admin sees all four tabs.
gcloud beta iap web get-iam-policy \
  --resource-type=cloud-run --service=documind-admin --region=us-central1